In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/iitm21f1002300/indic-lingua-data/val.jsonl
/kaggle/input/datasets/iitm21f1002300/indic-lingua-data/test.jsonl
/kaggle/input/datasets/iitm21f1002300/indic-lingua-data/train.jsonl


# Indic LLMLingua-2 Training — Optuna Hyperparameter Tuning

This notebook fine-tunes the LLMLingua-2 token classification model on an extended Indic dataset
(the original LLMLingua-2 dataset + additional XL-Sum supervision).

**Task:** binary token classification — each token is classified as `KEEP (1)` or `REMOVE (0)` during
prompt/context compression.

**Pipeline**
1. Load & inspect the dataset
2. Tokenize and align word-level labels to subword tokens
3. Define the compute-metrics function
4. Run an **Optuna** hyperparameter search (with real mid-training pruning)
5. Retrieve the best trial's hyperparameters
6. Re-train a final model end-to-end with those hyperparameters
7. Evaluate the final model on the **validation set** and save all artifacts

> Dataset used previously: 1,609 train / 201 validation / 202 test examples, class balance
> ≈ 78% REMOVE vs 22% KEEP tokens (523,634 vs 143,011). Base model: `FacebookAI/xlm-roberta-base`
> (~277M parameters).

## 1. Setup & Environment

In [2]:
!pip install -q optuna evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [3]:
import os
import gc
import json
import random

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import optuna

print("Torch      :", torch.__version__)
print("CUDA avail :", torch.cuda.is_available())
print("GPU count  :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

Torch      : 2.10.0+cu128
CUDA avail : True
GPU count  : 2
 - Tesla T4
 - Tesla T4


## 2. Experiment Configuration

Single source of truth for paths and hyperparameter constants — everything downstream reads from here.
A fixed random seed keeps data shuffling, weight init, and training behaviour reproducible.

In [4]:
# ==========================================================
# Paths
# ==========================================================
PROJECT_DIR = "/kaggle/working/indic_llmlingua"
DATASET_PATH = "/kaggle/input/datasets/iitm21f1002300/indic-lingua-data"

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
LOG_DIR        = os.path.join(PROJECT_DIR, "logs")
PREDICTION_DIR = os.path.join(PROJECT_DIR, "predictions")
CONFIG_DIR     = os.path.join(PROJECT_DIR, "configs")
TUNING_DIR     = os.path.join(PROJECT_DIR, "optuna_hyperparameter_search")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "final_model")

for directory in [
    PROJECT_DIR, CHECKPOINT_DIR, LOG_DIR,
    PREDICTION_DIR, CONFIG_DIR, TUNING_DIR, FINAL_MODEL_DIR,
]:
    os.makedirs(directory, exist_ok=True)

# ==========================================================
# Model / training constants
# ==========================================================
MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 512
NUM_LABELS = 2
SEED = 42

IGNORE_LABEL = -100

ID2LABEL = {0: "REMOVE", 1: "KEEP"}
LABEL2ID = {"REMOVE": 0, "KEEP": 1}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def fix_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fix_seed(SEED)

experiment_config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_labels": NUM_LABELS,
    "seed": SEED,
}
with open(os.path.join(CONFIG_DIR, "experiment_config.json"), "w") as fp:
    json.dump(experiment_config, fp, indent=4)

print(f"Device            : {DEVICE}")
print(f"Random seed fixed : {SEED}")
print("Project directories ready.")

Device            : cuda
Random seed fixed : 42
Project directories ready.


## 3. Load Dataset

Token-level binary labels for context compression, stored as JSONL. Each example contains
`question`, `original_answer`, `tokens`, and `labels`.

In [5]:
dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATASET_PATH}/train.jsonl",
        "validation": f"{DATASET_PATH}/val.jsonl",
        "test": f"{DATASET_PATH}/test.jsonl",
    },
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'original_answer', 'tokens', 'labels'],
        num_rows: 1609
    })
    validation: Dataset({
        features: ['question', 'original_answer', 'tokens', 'labels'],
        num_rows: 201
    })
    test: Dataset({
        features: ['question', 'original_answer', 'tokens', 'labels'],
        num_rows: 202
    })
})


In [6]:
# Class balance (flattened, ignoring padding/subword-continuation labels)
flat_labels = [
    label
    for sequence in dataset["train"]["labels"]
    for label in sequence
    if label != IGNORE_LABEL
]
class_counts = np.bincount(flat_labels, minlength=2)

print("Train / Validation / Test samples:",
      len(dataset["train"]), "/", len(dataset["validation"]), "/", len(dataset["test"]))
print("Class counts [REMOVE, KEEP]:", class_counts)

Train / Validation / Test samples: 1609 / 201 / 202
Class counts [REMOVE, KEEP]: [523634 143011]


## 4. Tokenizer & Label Alignment

XLM-RoBERTa uses SentencePiece, so a single word may split into multiple subword tokens. Only the
**first** subword of each word keeps its original label; all following subwords (and special /
padding tokens) get `-100` so they're ignored in the loss.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: FacebookAI/xlm-roberta-base


In [8]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for batch_index in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        labels = examples["labels"][batch_index]

        previous_word = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:                       # special / padding tokens
                label_ids.append(IGNORE_LABEL)
            elif word_id != previous_word:              # first subword of a word
                label_ids.append(labels[word_id])
            else:                                        # continuation subwords
                label_ids.append(IGNORE_LABEL)
            previous_word = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset",
)

tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(tokenized_dataset)

Tokenizing dataset:   0%|          | 0/1609 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/201 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/202 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1609
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 201
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 202
    })
})


## 5. Data Collator & Metrics

Dynamic padding per batch (instead of padding every example to `MAX_LENGTH`) improves GPU
utilization. Labels are padded with `-100` so padded positions never contribute to the loss.

In [9]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions, true_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        for pred, label in zip(pred_seq, label_seq):
            if label == IGNORE_LABEL:
                continue
            true_predictions.append(pred)
            true_labels.append(label)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average="binary", zero_division=0,
    )
    accuracy = accuracy_score(true_labels, true_predictions)

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


print("Data collator and compute_metrics ready.")

Data collator and compute_metrics ready.


## 6. Model Builder

A fresh XLM-RoBERTa encoder + randomly-initialized 2-class token classification head
(`0 = REMOVE`, `1 = KEEP`) is instantiated for every trial, so experiments never leak weights
into one another.

In [10]:
def build_model():
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    model.gradient_checkpointing_enable()
    model.to(DEVICE)
    return model


_probe_model = build_model()
total_params = sum(p.numel() for p in _probe_model.parameters())
trainable_params = sum(p.numel() for p in _probe_model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

del _probe_model
gc.collect()
torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters     : 277,454,594
Trainable parameters : 277,454,594


## 7. Hyperparameter Tuning with Optuna

For every trial, Optuna samples: `learning_rate`, `per_device_train_batch_size`,
`gradient_accumulation_steps`, `weight_decay`, `warmup_ratio`, `num_train_epochs`,
`lr_scheduler_type`, and `max_grad_norm`.

A fresh model is trained on the training split and evaluated on the **validation** split after
every epoch. An `OptunaPruningCallback` reports the running validation F1 back to Optuna after
each epoch and stops the trial early if it's clearly underperforming — this is what actually makes
`MedianPruner` useful (the previous version created a pruner but never called `trial.report`, so
every trial ran to completion regardless of how bad it looked early on).

The best trial's F1 score is returned as the objective to maximize.

In [11]:
class OptunaPruningCallback(TrainerCallback):
    """
    Reports validation F1 to Optuna after each evaluation
    and prunes unpromising trials.
    """

    def __init__(self, trial, metric_name="eval_f1"):
        self.trial = trial
        self.metric_name = metric_name

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):

        if metrics is None:
            return control

        score = metrics.get(self.metric_name)

        if score is None:
            return control

        epoch = int(round(state.epoch or 0))

        self.trial.report(score, step=epoch)

        if self.trial.should_prune():
            raise optuna.TrialPruned(
                f"Trial {self.trial.number} pruned at epoch "
                f"{state.epoch:.1f} (f1={score:.4f})"
            )

        return control


def objective(trial):

    # ==========================================================
    # 1. OPTUNA HYPERPARAMETERS
    # ==========================================================

    learning_rate = trial.suggest_float(
        "learning_rate",
        5e-6,
        5e-5,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "per_device_train_batch_size",
        [8, 16]
    )

    gradient_accumulation_steps = trial.suggest_categorical(
        "gradient_accumulation_steps",
        [1, 2, 4]
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        0.1,
        log=True
    )

    warmup_ratio = trial.suggest_float(
        "warmup_ratio",
        0.0,
        0.15
    )

    num_train_epochs = trial.suggest_int(
        "num_train_epochs",
        2,
        6
    )

    lr_scheduler_type = trial.suggest_categorical(
        "lr_scheduler_type",
        [
            "linear",
            "cosine",
            "cosine_with_restarts"
        ]
    )

    max_grad_norm = trial.suggest_float(
        "max_grad_norm",
        0.5,
        2.0
    )


    # ==========================================================
    # 2. PRINT TRIAL CONFIGURATION
    # ==========================================================

    config = {
        "learning_rate": learning_rate,
        "per_device_train_batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "weight_decay": weight_decay,
        "warmup_ratio": warmup_ratio,
        "num_train_epochs": num_train_epochs,
        "lr_scheduler_type": lr_scheduler_type,
        "max_grad_norm": max_grad_norm,
    }

    print("\n" + "=" * 80)
    print(f"OPTUNA TRIAL {trial.number}")
    print("=" * 80)

    for key, value in config.items():
        print(f"{key}: {value}")

    print("=" * 80)


    # ==========================================================
    # 3. REPRODUCIBILITY
    # ==========================================================

    fix_seed(SEED)


    # ==========================================================
    # 4. TRIAL OUTPUT DIRECTORY
    # ==========================================================

    trial_output_dir = os.path.join(
        TUNING_DIR,
        f"trial_{trial.number}"
    )

    os.makedirs(
        trial_output_dir,
        exist_ok=True
    )


    # ==========================================================
    # 5. TRAINING ARGUMENTS
    # ==========================================================

    training_args = TrainingArguments(

        output_dir=trial_output_dir,

        # -------------------------
        # Training
        # -------------------------
        num_train_epochs=num_train_epochs,

        learning_rate=learning_rate,

        per_device_train_batch_size=batch_size,

        per_device_eval_batch_size=8,

        gradient_accumulation_steps=gradient_accumulation_steps,

        weight_decay=weight_decay,

        warmup_ratio=warmup_ratio,

        lr_scheduler_type=lr_scheduler_type,

        max_grad_norm=max_grad_norm,

        # -------------------------
        # Evaluation
        # -------------------------
        eval_strategy="epoch",

        # Trials are temporary.
        # Final best model will be trained separately.
        save_strategy="no",

        # -------------------------
        # LOGGING
        # -------------------------

        # IMPORTANT:
        # Log training loss once per epoch
        logging_strategy="epoch",

        # IMPORTANT:
        # Show the Trainer progress bar
        disable_tqdm=False,

        # -------------------------
        # Best metric
        # -------------------------

        metric_for_best_model="f1",

        greater_is_better=True,

        # -------------------------
        # Performance
        # -------------------------

        fp16=torch.cuda.is_available(),

        # -------------------------
        # Reproducibility
        # -------------------------

        seed=SEED,

        # -------------------------
        # Disable external logging
        # -------------------------

        report_to="none",
    )


    # ==========================================================
    # 6. CREATE FRESH MODEL
    # ==========================================================

    model = build_model()


    # ==========================================================
    # 7. CREATE TRAINER
    # ==========================================================

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=tokenized_dataset["train"],

        eval_dataset=tokenized_dataset["validation"],

        processing_class=tokenizer,

        data_collator=data_collator,

        compute_metrics=compute_metrics,

        callbacks=[
            OptunaPruningCallback(trial),

            EarlyStoppingCallback(
                early_stopping_patience=2,
                early_stopping_threshold=0.0
            ),
        ],
    )


    # ==========================================================
    # 8. TRAIN TRIAL
    # ==========================================================

    try:

        trainer.train()


        # ======================================================
        # 9. FINAL EVALUATION OF THIS TRIAL
        # ======================================================

        metrics = trainer.evaluate()


        # ======================================================
        # 10. EXTRACT BEST F1
        # ======================================================

        f1 = metrics.get("eval_f1")

        if f1 is None:
            raise ValueError(
                "eval_f1 was not found in evaluation metrics."
            )


        # ======================================================
        # 11. STORE OPTUNA USER ATTRIBUTES
        # ======================================================

        trial.set_user_attr(
            "eval_accuracy",
            metrics.get("eval_accuracy")
        )

        trial.set_user_attr(
            "eval_precision",
            metrics.get("eval_precision")
        )

        trial.set_user_attr(
            "eval_recall",
            metrics.get("eval_recall")
        )

        trial.set_user_attr(
            "eval_loss",
            metrics.get("eval_loss")
        )


        # ======================================================
        # 12. CREATE EPOCH-WISE TRAINING TABLE
        # ======================================================

        history = trainer.state.log_history

        training_logs = [
            log for log in history
            if "loss" in log and "eval_loss" not in log
        ]

        evaluation_logs = [
            log for log in history
            if "eval_loss" in log
        ]


        epoch_results = []

        for eval_log in evaluation_logs:

            epoch = eval_log.get("epoch")

            if epoch is None:
                continue

            # Find training log corresponding to this epoch
            matching_train_logs = [
                log for log in training_logs
                if log.get("epoch") is not None
                and abs(
                    float(log["epoch"]) - float(epoch)
                ) < 0.01
            ]

            train_loss = None

            if matching_train_logs:
                train_loss = matching_train_logs[-1].get("loss")


            epoch_results.append({

                "Epoch": int(round(epoch)),

                "Training Loss": train_loss,

                "Validation Loss": eval_log.get(
                    "eval_loss"
                ),

                "Accuracy": eval_log.get(
                    "eval_accuracy"
                ),

                "Precision": eval_log.get(
                    "eval_precision"
                ),

                "Recall": eval_log.get(
                    "eval_recall"
                ),

                "F1": eval_log.get(
                    "eval_f1"
                ),
            })


        # ======================================================
        # 13. DISPLAY EPOCH-WISE RESULTS
        # ======================================================

        if epoch_results:

            history_df = pd.DataFrame(
                epoch_results
            )

            print("\n")
            print(history_df.to_string(index=False))

        else:

            print(
                "\nWarning: No epoch-wise training "
                "history was found."
            )


        # ======================================================
        # 14. DISPLAY FINAL TRIAL METRICS
        # ======================================================

        print("\n" + "-" * 80)
        print(f"TRIAL {trial.number} COMPLETED")
        print("-" * 80)

        print(
            f"Validation Loss : "
            f"{metrics.get('eval_loss', float('nan')):.6f}"
        )

        print(
            f"Accuracy        : "
            f"{metrics.get('eval_accuracy', float('nan')):.6f}"
        )

        print(
            f"Precision       : "
            f"{metrics.get('eval_precision', float('nan')):.6f}"
        )

        print(
            f"Recall          : "
            f"{metrics.get('eval_recall', float('nan')):.6f}"
        )

        print(
            f"F1              : "
            f"{f1:.6f}"
        )

        print("-" * 80)


        # ======================================================
        # 15. RETURN OPTUNA OBJECTIVE
        # ======================================================

        return f1


    except optuna.TrialPruned:

        print(
            f"\nTrial {trial.number} was pruned."
        )

        raise


    except Exception as exc:

        print(
            f"\nTrial {trial.number} failed:"
        )

        print(exc)

        raise


    finally:

        # ======================================================
        # 16. GPU MEMORY CLEANUP
        # ======================================================

        del trainer
        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [12]:
N_TRIALS = 30          # lower this (e.g. 10-15) for a quicker search
TIMEOUT_SECONDS = 36000  # e.g. 6 * 3600 to hard-cap the search to 6 hours

study = optuna.create_study(
    study_name="xlmr_prompt_compression",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=TIMEOUT_SECONDS,
    gc_after_trial=True,
)

[I 2026-08-25 18:26:07,003] A new study created in memory with name: xlmr_prompt_compression
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



OPTUNA TRIAL 0
learning_rate: 1.1844319751820392e-05
per_device_train_batch_size: 8
gradient_accumulation_steps: 1
weight_decay: 0.00014936568554617635
warmup_ratio: 0.12992642186624026
num_train_epochs: 5
lr_scheduler_type: cosine_with_restarts
max_grad_norm: 1.7486639612006325


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.12/dist-packages/torch/au

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.041590,0.931387,0.729806,0.548956,0.264860,0.357320
2,0.889549,0.933204,0.736040,0.664210,0.139997,0.231252
3,0.863532,0.906671,0.750579,0.606230,0.343806,0.438774
4,0.826058,0.924241,0.748801,0.582881,0.401634,0.475574
5,0.798424,0.922802,0.742975,0.545343,0.563324,0.554188


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.12/dist-packages/torch/au

Epoch,Training Loss,Validation Loss


[I 2026-08-25 20:49:30,151] Trial 29 pruned. Trial 29 pruned at epoch 1.0 (f1=0.0000)



Trial 29 was pruned.


In [13]:
print("\n" + "=" * 80)
print("BEST TRIAL")
print("=" * 80)
print(f"Trial number : {study.best_trial.number}")
print(f"Best val F1   : {study.best_value}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

BEST_PARAMS = study.best_params


BEST TRIAL
Trial number : 0
Best val F1   : 0.5541875406866491

Best hyperparameters:
  learning_rate: 1.1844319751820392e-05
  per_device_train_batch_size: 8
  gradient_accumulation_steps: 1
  weight_decay: 0.00014936568554617635
  warmup_ratio: 0.12992642186624026
  num_train_epochs: 5
  lr_scheduler_type: cosine_with_restarts
  max_grad_norm: 1.7486639612006325


### Saving Optuna results

In [14]:
df_trials = study.trials_dataframe()
df_trials.to_csv(os.path.join(TUNING_DIR, "optuna_trials.csv"), index=False)

with open(os.path.join(CONFIG_DIR, "best_hyperparameters.json"), "w") as fp:
    json.dump(BEST_PARAMS, fp, indent=4)

print(f"Saved {len(df_trials)} trial(s) to {os.path.join(TUNING_DIR, 'optuna_trials.csv')}")
print(f"Saved best hyperparameters to {os.path.join(CONFIG_DIR, 'best_hyperparameters.json')}")

Saved 30 trial(s) to /kaggle/working/indic_llmlingua/optuna_hyperparameter_search/optuna_trials.csv
Saved best hyperparameters to /kaggle/working/indic_llmlingua/configs/best_hyperparameters.json


## 8. Final Training Using the Best Hyperparameters

Retrains a fresh model on the full training split using `study.best_params`, this time with
checkpointing enabled (`save_strategy="epoch"`, `load_best_model_at_end=True`) so the
best-on-validation checkpoint is automatically restored at the end of training.

In [15]:
fix_seed(SEED)

final_training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    num_train_epochs=BEST_PARAMS["num_train_epochs"],
    learning_rate=BEST_PARAMS["learning_rate"],
    per_device_train_batch_size=BEST_PARAMS["per_device_train_batch_size"],
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=BEST_PARAMS["gradient_accumulation_steps"],
    weight_decay=BEST_PARAMS["weight_decay"],
    warmup_ratio=BEST_PARAMS["warmup_ratio"],
    lr_scheduler_type=BEST_PARAMS["lr_scheduler_type"],
    max_grad_norm=BEST_PARAMS["max_grad_norm"],

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=20,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

final_model = build_model()

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0)],
)

final_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and re

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.875674,0.931387,0.729806,0.548956,0.264860,0.357320
2,0.928700,0.933204,0.736040,0.664210,0.139997,0.231252
3,0.891907,0.906647,0.750563,0.606147,0.343863,0.438799
4,0.837707,0.922175,0.752276,0.593310,0.402094,0.479336
5,0.851601,0.919517,0.745341,0.551394,0.547270,0.549324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=505, training_loss=0.8844920715483109, metrics={'train_runtime': 709.0412, 'train_samples_per_second': 11.346, 'train_steps_per_second': 0.712, 'total_flos': 2102132407941120.0, 'train_loss': 0.8844920715483109, 'epoch': 5.0})

## 9. Evaluate the Best Model on the Validation Set

In [16]:
val_metrics = final_trainer.evaluate(tokenized_dataset["validation"])

print("\n" + "=" * 80)
print("FINAL MODEL — VALIDATION METRICS")
print("=" * 80)
for key, value in val_metrics.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



FINAL MODEL — VALIDATION METRICS
eval_loss: 0.9195170998573303
eval_accuracy: 0.7453412094905518
eval_precision: 0.5513942837265928
eval_recall: 0.5472696933080154
eval_f1: 0.549324246274691
eval_runtime: 4.5285
eval_samples_per_second: 44.385
eval_steps_per_second: 2.871
epoch: 5.0


## 10. Save the Final Model & Artifacts

In [17]:
final_trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

with open(os.path.join(FINAL_MODEL_DIR, "validation_metrics.json"), "w") as fp:
    json.dump(val_metrics, fp, indent=4)

with open(os.path.join(FINAL_MODEL_DIR, "best_hyperparameters.json"), "w") as fp:
    json.dump(BEST_PARAMS, fp, indent=4)

history_df = pd.DataFrame(final_trainer.state.log_history)
history_df.to_csv(os.path.join(FINAL_MODEL_DIR, "training_history.csv"), index=False)

print(f"Final model, tokenizer, metrics and training history saved to: {FINAL_MODEL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model, tokenizer, metrics and training history saved to: /kaggle/working/indic_llmlingua/final_model
